# Volve Field - Notebook 1: Cleaning the Data
This notebook does in Python exactly whats in Excel:
load the raw production data, tidy it up, flag each well as a producer or
injector, and build a few useful columns (KPIs).

**Data:** Equinor Volve open dataset (CC BY-NC-SA 4.0) — Daily Production sheet.

The daily sheet is used because it has more detail (pressure, choke) than the monthly sheet, which is better suited to Excel dashboards.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/Volve_Production_Data.xlsx", sheet_name="Daily Production Data") # define the dataframe df

df.head() # show first 5 rows

df.shape # (number of rows, number of columns)


In [ ]:
# Rename the columns to plain simple names 
# `rename` takes a dictionary: "old name": "new name"

df = df.rename(columns={
    "DATEPRD": "Date",
    "WELL_BORE_CODE": "Well",
    "ON_STREAM_HRS": "On Stream Hours",
    "BORE_OIL_VOL": "Oil",
    "BORE_GAS_VOL": "Gas",
    "BORE_WAT_VOL": "Water",
    "BORE_WI_VOL": "Water Injected",
    "FLOW_KIND": "Flow Kind",
    "WELL_TYPE": "Well Type",
    "AVG_DOWNHOLE_PRESSURE": "Downhole Pressure",
    "AVG_CHOKE_SIZE_P": "Choke Size",
    "AVG_WHP_P": "Wellhead Pressure"
})

df.head()

In [ ]:
# Flag each row as Producer or Injector

# set every row to "Producer". Then, for the rows where `Flow Kind` is
# "injection", change it to "Injector". add column well role to the dataframe

df["Well Role"] = "Producer"
df.loc[df["Flow Kind"] == "injection", "Well Role"] = "Injector"

df["Well Role"].value_counts()

#df.head()


In [ ]:
# injector rows had "NULL" in the oil/gas/water columns, replaced with 0. affects averages later hence "On Stream Hours" used to filter

df["Oil"] = df["Oil"].fillna(0)
df["Gas"] = df["Gas"].fillna(0)
df["Water"] = df["Water"].fillna(0)
df["Water Injected"] = df["Water Injected"].fillna(0)

df[["Oil", "Gas", "Water", "Water Injected"]].isnull().sum()

In [ ]:
# Build the KPI columns as same calc in excel. Making a new column in pandas is just: `df["New Column"] = a calculation`
# 
# Oil Rate (Sm³/day) = oil per producing day. 
# On Stream Hours / 24 converts hours to days, matching the daily rates NPD reports.

df["Oil Rate"] = df["Oil"] / (df["On Stream Hours"] / 24) # Oil Rate (Sm³/day)

df["Water Cut"] = df["Water"] / (df["Oil"] + df["Water"]) # Water Cut (fraction/percentage)

df["GOR"] = df["Gas"] / df["Oil"] # GOR (Gas-Oil Ratio) = gas produced per unit of oil.

df = df.replace([np.inf, -np.inf], np.nan) # np.nan is pd word for "blank").

df[["Date", "Well", "Well Role", "Oil", "Oil Rate", "Water Cut", "GOR"]].head()

#df.head() # check with top excel row in data sheet


In [ ]:
df.to_csv("volve_daily_clean.csv", index=False)
print("Saved. Rows:", df.shape[0], "Columns:", df.shape[1])

In [ ]:
# import os
# print(os.getcwd())